# SAGE and Global Feature Ranking

This notebook demonstrates **SAGE** (Shapley Additive Global importancE) — a Shapley-based
method for global feature importance that properly accounts for feature interactions.

We also show how to:
- Compare SAGE with SHAP-based importance and permutation importance side-by-side
- Group features for grouped SAGE and grouped SHAP

SAGE requires the optional `sage-importance` package:
```
pip install sage-importance
```

**Reference:** Covert, I., Lundberg, S., and Lee, S.-I., 2020: Understanding Global Feature
Contributions With Additive Importance Measures. NeurIPS.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

import skexplain

## Create Synthetic Dataset

A weather-inspired binary classification task with known feature importance structure.

In [ ]:
np.random.seed(42)
N = 2000
X = pd.DataFrame({
    'CAPE': np.random.exponential(1500, N),
    'Shear': np.random.gamma(3, 5, N),
    'Freezing_Lvl': 2500 + np.random.randn(N) * 500,
    'Moisture': np.random.beta(3, 2, N) * 20,
    'Temperature': 25 + np.random.randn(N) * 8,
    'Noise': np.random.randn(N) * 10,
})

logit = 0.002*X['CAPE'] + 0.08*X['Shear'] - 0.001*X['Freezing_Lvl'] + 0.05*X['Moisture'] - 5.0
y = (np.random.rand(N) < 1/(1+np.exp(-logit))).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test = X_test.reset_index(drop=True)

gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
print(f'Test accuracy: {gb.score(X_test, y_test):.3f}')

## Compute SAGE Values

In [ ]:
explainer = skexplain.ExplainToolkit(
    estimators=[('GB', gb)],
    X=X_test, y=y_test,
)

sage_results = explainer.sage(n_background=50, n_jobs=1)
print('SAGE rankings:', sage_results['sage_rankings__GB'].values)
print(f'Computation time: {sage_results.attrs["computation_time_seconds"]}s')

## Plot SAGE Importance

In [ ]:
fig, axes = explainer.plot_importance(
    data=sage_results,
    panels=[('sage', 'GB')],
)

## Compare SAGE vs SHAP vs Permutation Importance

Compute all three global ranking methods and plot them side-by-side.

In [ ]:
# Compute SHAP and convert to importance
shap_results = explainer.local_attributions(method='shap')
shap_importance = skexplain.to_skexplain_importance(
    shap_results['shap_values__GB'].values,
    estimator_name='GB',
    feature_names=list(X_test.columns),
    method='shap_sum',
)

# Compute permutation importance
perm_imp = explainer.permutation_importance(
    n_vars=6, evaluation_fn='auc', n_permute=5,
)

In [ ]:
# Plot all three side by side
fig, axes = explainer.plot_importance(
    data=[sage_results, shap_importance, perm_imp],
    panels=[
        ('sage', 'GB'),
        ('shap_sum', 'GB'),
        ('backward_multipass', 'GB'),
    ],
)

## SHAP Summary + SAGE Side-by-Side

A common visualization pattern: SHAP beeswarm (feature relevance) next to SAGE bars (feature importance).

In [ ]:
fig, axes = plt.subplots(dpi=300, ncols=2, figsize=(12, 6))

# Left panel: SHAP summary plot
explainer.scatter_plot(
    dataset=shap_results,
    estimator_name='GB',
    method='shap',
    plot_type='summary',
    ax=axes[0],
    fig=fig,
    add_colorbar=False,
    max_display=6,
)
axes[0].set_xlabel('SHAP Value')
axes[0].set_title('Feature Relevance (SHAP)', fontsize=12)

# Right panel: SAGE importance
explainer.plot_importance(
    data=sage_results,
    panels=[('sage', 'GB')],
    ax=axes[1],
    xlabels=['SAGE Value'],
    show_method_subtitle=False,
)
axes[1].set_title('Feature Importance (SAGE)', fontsize=12)

plt.tight_layout()

## Grouped SAGE

Group features into categories and compute SAGE at the group level.
This can be done in two ways:
1. **Directly** via `explainer.sage(groups=...)` — groups features during computation
2. **Post-hoc** via `skexplain.group_sage(sage_results, groups)` — sums individual SAGE values

In [ ]:
groups = {
    'Thermodynamic': ['CAPE', 'Moisture', 'Temperature'],
    'Kinematic': ['Shear'],
    'Environmental': ['Freezing_Lvl', 'Noise'],
}

# Method 1: Direct grouped computation
grouped_sage = explainer.sage(groups=groups, n_background=50, n_jobs=1)
print('Direct grouped rankings:', grouped_sage['grouped_sage_rankings__GB'].values)

# Method 2: Post-hoc grouping
grouped_post = skexplain.group_sage(sage_results, groups, estimator_name='GB')
print('Post-hoc grouped rankings:', grouped_post['grouped_sage_rankings__GB'].values)

In [ ]:
fig, axes = explainer.plot_importance(
    data=[grouped_sage, grouped_post],
    panels=[
        ('grouped_sage', 'GB'),
        ('grouped_sage', 'GB'),
    ],
    xlabels=['Direct Grouped SAGE', 'Post-hoc Grouped SAGE'],
)

## Grouped SHAP

You can also group SHAP values for grouped beeswarm plots using the
built-in utilities `group_local_values` and `group_feature_values`.

In [ ]:
# Group SHAP values
X_grouped = skexplain.group_feature_values(X_test, groups)
grouped_shap = skexplain.group_local_values(shap_results, groups, X_grouped)

# Create explainer with grouped features
explainer_grouped = skexplain.ExplainToolkit(X=X_grouped)

# Plot grouped SHAP summary
explainer_grouped.scatter_plot(
    dataset=grouped_shap,
    estimator_name='GB',
    method='shap',
    plot_type='summary',
)